# Retake 2025/26 task 1: Fine-tuning with cached embeddings

In this task, we train a simple ConvNet on FashionMNIST (10-class classification of grayscale 28×28 images).<br>
We start with a normal training, but cut it short after one epoch, resulting in a roughly pretrained model.

Your task is to continue the training by fine-tuning only the last few layers of the model for a few epochs;<br>
but to speed that up, you should cache the embedding outputted by the previous layers.<br>
More precisely, do the following:
1. In `FineTuner.prepare_embeddings_cache`: compute the output of the backbone (the first `num_initial_layers` layers of the model) for all samples, and store them in memory with dtype `torch.bfloat16`.
2. `FineTuner.prepare_embeddings_cache` should return the memory usage – the number of bytes used by the cached data (just the tensor data, disregard any tensor metadata and Python object overhead).
3. In `FineTuner.finetune`: use the cached embeddings to train the head (the remaining layers of the model) for `num_epochs`, and return the test accuracies achieved after each epoch.
4. Instead of using typical augmentations, we can try adding noise to cached embeddings. Adapt your code so that `finetune` uses embeddings plus normal noise times `noise_level` as an augmentation. 
5. Plot a comparison of accuracies over time for different noise levels: `[0, 0.02, 0.2]` (a colored line plot with labeled axes and a legend, nothing more). Do not expect any improvements from adding noise, it's just an example.

Notes:
* You can get partial credit for completing only some of the steps (in fact, we expect that very few people will complete everything).
* You can assume that the `pretrained_model` is an `nn.Sequential` instance where each step takes a single tensor and outputs a single tensor.
* Your memory usage doesn't have to optimal: for example, extra copies of labels won't be penalized.
* Do not change hyperparameters (the number of epochs, the learning rate, batch size, etc.)
* Do not change devices (cpu only) or dtypes.
* You can add imports if you want to.

Hints:
* `dataset = TensorDataset(a, b, c)` is a dataset such that `dataset[i] == (a[i], b[i], c[i])`
* `ConcatDataset([d1, d2, ...])` converts a sequence of datasets into one dataset containing all their elements.
* Any `Sequence` can be used as a dataset (any class that implements `def __len__(self) -> int:` and `def __getitem__(self, index: int) -> Any:`).

In [ ]:
!tar -xf retake-task1-data.tar.gz

In [ ]:
import copy
from typing import TypedDict

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
from torch.utils.data import Dataset, DataLoader, TensorDataset, ConcatDataset
from torchvision import datasets
from torchvision.ops.misc import Conv2dNormActivation
from torchvision.transforms import v2
from tqdm import tqdm

## Normal training (do not change anything in this section)

In [ ]:
torch.manual_seed(1);

In [ ]:
def train_epoch(
    model: nn.Module,
    train_loader: DataLoader,
    optimizer: optim.Optimizer,
    epoch: int,
    quiet: bool = False,
) -> None:
    model.train()
    pbar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch}",
        disable=quiet,
    )
    for _batch_idx, (data, target) in pbar:
        # data, target = data.to(device), target.to(device)
        data = data.to(torch.float32)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())


def test(model: nn.Module, test_loader: DataLoader, quiet: bool = False) -> float:
    """Returns the accuracy (0..1)."""
    model.eval()
    test_loss, n_correct, n_total = 0.0, 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            # data, target = data.to(device), target.to(device)
            data = data.to(torch.float32)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction="sum").item()
            pred = output.argmax(dim=1)
            n_correct += (pred == target).sum().item()
            n_total += data.shape[0]

    test_loss /= n_total
    if not quiet:
        print(
            f"Test loss: {test_loss:.4f}, accuracy: {n_correct}/{n_total} ({n_correct / n_total:.1%})"
        )
    return n_correct / n_total


class DataloaderArgs(TypedDict, total=False):
    batch_size: int
    shuffle: bool
    num_workers: int


train_kwargs = DataloaderArgs(batch_size=256, num_workers=2, shuffle=True)
test_kwargs = DataloaderArgs(batch_size=1000, num_workers=2, shuffle=False)


def train(
    model: nn.Module,
    train_dataset: Dataset,
    test_dataset: Dataset,
    num_epochs: int = 1,
    learning_rate: float = 1e-2,
    quiet: bool = False,
) -> list[float]:
    """Returns a list of self.num_epochs test accuracies."""
    train_loader = DataLoader(train_dataset, **train_kwargs)
    test_loader = DataLoader(test_dataset, **test_kwargs)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

    test_losses = list[float]()
    for epoch in range(1, num_epochs + 1):
        train_epoch(model, train_loader, optimizer, epoch, quiet)
        test_losses.append(test(model, test_loader, quiet))
    return test_losses

In [ ]:
transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize((0.1307,), (0.3081,)),
    ]
)
train_dataset = datasets.FashionMNIST(
    "./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST("./data", train=False, transform=transform)

In [ ]:
pretrained_model = nn.Sequential(
    Conv2dNormActivation(1, 32, stride=2),
    Conv2dNormActivation(32, 32, stride=2),
    Conv2dNormActivation(32, 32),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(32, 10),
)
num_initial_layers=4
train(pretrained_model, train_dataset, test_dataset)

## FineTuner

In [ ]:
### YOUR CODE HERE (optional) ###
### === ###


class FineTuner:
    def __init__(
        self,
        model: nn.Sequential,
        train_loader: DataLoader,
        test_loader: DataLoader,
        num_initial_layers: int = num_initial_layers,
        num_epochs: int = 5,
        learning_rate: float = 1e-2,
    ) -> None:
        self.backbone: nn.Sequential = model[:num_initial_layers]
        self.head: nn.Sequential = model[num_initial_layers:]
        self.orig_train_loader = train_loader
        self.orig_test_loader = test_loader
        self.num_epochs = num_epochs
        self.learning_rate = learning_rate
        ### YOUR CODE HERE (optional) ###
        ### === ###

    ### YOUR CODE HERE (optional) ###
    ### === ###

    def prepare_embeddings_cache(self) -> int:
        """Returns the number of bytes used by data in the cache."""
        ### YOUR CODE HERE ###

        ### === ###

    def finetune(self, noise_level: float = 0.0) -> list[float]:
        """Returns a list of self.num_epochs test accuracies."""
        ### YOUR CODE HERE ###


        ### === ###

In [ ]:
model = copy.deepcopy(pretrained_model)
fine_tuner = FineTuner(
    model,
    num_initial_layers=num_initial_layers,
    train_loader=DataLoader(train_dataset, **train_kwargs),
    test_loader=DataLoader(test_dataset, **test_kwargs),
)
n_bytes = fine_tuner.prepare_embeddings_cache()
print(f"Memory used by cache: {n_bytes / 1024 / 1024:.3f} MiB")
print(fine_tuner.finetune())

## Compare fine-tuning with different noise levels.

In [ ]:
accuracies = dict[float, list[float]]()
noise_levels = [0, 0.02, 0.2]
### YOUR CODE HERE ###
### === ###

plt.figure(figsize=(10, 6))
for noise_level, acc_list in accuracies.items():
    epochs = range(1, len(acc_list) + 1)
    plt.plot(epochs, acc_list, marker="o", label=f"noise={noise_level}")

plt.xlabel("Epoch")
plt.ylabel("Test accuracy")
plt.legend()
plt.grid(True)
plt.show()